# Part 3: Model Explainability with SHAP (SHapley Additive exPlanations)
### Project: AI-Powered Agentic Smart Campus Intelligence & Student Success Platform
**Author:** Sumith U (SRN: R24EA807) | 7th Sem B.Tech CSE (AI & ML), REVA University  
**Specification:** Part 3 SHAP Explainability Workflow  

---

## 1. Objective of Part 3
The goal of this phase is to make our student risk prediction model transparent and explainable:
1. **Global Explainability:** Understand what features drive predictions across all students (Mean Absolute SHAP and Beeswarm plots).
2. **Local Explainability:** Explain individual student predictions with waterfall plots showing exact positive and negative feature contributions.
3. **Real-Time Advisor Decision Support:** Enable academic mentors to identify specific actionable factors behind high-risk alerts.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib

# Add project root to sys.path
base_dir = Path.cwd().parent
sys.path.insert(0, str(base_dir))

from src.explain_model import (
    load_model_and_test_data,
    compute_shap_explanations,
    explain_single_raw_student
)

sns.set_theme(style="whitegrid")
%matplotlib inline
print(f"SHAP version: {shap.__version__} loaded successfully.")


## 2. Load Model & Compute Shapley Additive Values
We compute SHAP values using the exact linear formulation on 250 evaluation students from our holdout test set.


In [ ]:
shap_values, explainer, y_test_proba, high_idx, low_idx = compute_shap_explanations(sample_size=250)

print(f"SHAP Values Matrix Shape: {shap_values.shape}")
print(f"Model Base Value (phi_0): {explainer.expected_value:.4f} log-odds")
print(f"High-Risk Student #{high_idx + 1} Dropout Probability: {y_test_proba[high_idx]:.2%}")
print(f"Low-Risk Student  #{low_idx + 1} Dropout Probability: {y_test_proba[low_idx]:.2%}")


## 3. Global Feature Importance Ranking (Mean Absolute SHAP)
The bar plot below ranks the most influential features across all students.


In [ ]:
plt.figure(figsize=(9, 6))
shap.plots.bar(shap_values, max_display=12, show=True)


## 4. Global Beeswarm Plot (Directionality & Feature Values)
Each dot represents a student. Red indicates a high feature value, and blue indicates a low feature value.
* Points to the **right** increase dropout risk.
* Points to the **left** decrease dropout risk.


In [ ]:
plt.figure(figsize=(10, 6.5))
shap.plots.beeswarm(shap_values, max_display=12, show=True)


## 5. Local Student Case Breakdown (Waterfall Plots)
Let us inspect the individual feature attributions for our high-risk student and our low-risk student.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

print(f"--- At-Risk Case #{high_idx + 1} Waterfall ---")
shap.plots.waterfall(shap_values[high_idx], max_display=10, show=True)

print(f"--- Persistent Case #{low_idx + 1} Waterfall ---")
shap.plots.waterfall(shap_values[low_idx], max_display=10, show=True)


## 6. Real-Time Advisor Tool: Explain Any Student Record
Faculty mentors can feed any raw, un-preprocessed student record to get an instant breakdown of risk drivers.


In [ ]:
raw_df = pd.read_csv(base_dir / "data" / "raw" / "data.csv", sep=";").drop(columns=["Target"])
sample_student = raw_df.iloc[[0]]

explanation = explain_single_raw_student(sample_student)

print(f"Predicted Dropout Probability: {explanation['predicted_dropout_probability']}%")
print(f"Assigned Risk Tier:            {explanation['risk_tier']}")
print("\nTop Risk Escalators (Pushing Risk Higher):")
for d in explanation['top_5_positive_risk_drivers']:
    print(f"  + {d['feature']}: SHAP = +{d['shap_value']:.4f}")

print("\nTop Protective Factors (Shielding Student from Risk):")
for d in explanation['top_5_protective_factors']:
    print(f"  - {d['feature']}: SHAP = {d['shap_value']:.4f}")


## 7. Summary & Next Steps (Ready for Part 4)
* **Global Attributions:** Curricular unit approval and tuition fee status are the strongest predictors.
* **Local Attributions:** High-risk students exhibit zero approved credits in semester 2 and unpaid tuition.
* **Responsible AI:** SHAP attributions represent model correlation, not real-world causality.

**Ready for Part 4: Prerequisite Knowledge Graph Construction & Cognitive Mastery Diagnosis.**
